# LeetCode 3199 — Count Triplets with Even XOR Set Bits I bằng Python

Link đề: https://leetcode.com/problems/count-triplets-with-even-xor-set-bits-i/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Array (đếm theo tính chẵn lẻ)

Skeleton LeetCode:

```python
class Solution:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        ...
```

Ghi chú: đây là bài **Premium**. Đề và các ví dụ trong notebook được viết theo hiểu biết về bài; mọi đáp án đã được kiểm chứng bằng code, nhưng nếu có tài khoản Premium hãy **đối chiếu lại đề gốc** (nhất là ràng buộc và câu chữ của ví dụ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho ba mảng `a`, `b`, `c`; đếm số **bộ ba** `(a[i], b[j], c[k])` mà **XOR** của ba số đó có **số bit 1 (set bits) là số chẵn**.
2. Nắm các công cụ Python/bit: toán tử XOR `^`, đếm bit 1 bằng `bin(x).count("1")` hoặc `x.bit_count()`, kiểm tra chẵn lẻ bằng `& 1` / `% 2`, đếm bằng list hai phần tử hoặc `Counter`, và `itertools.product` để liệt kê tổ hợp.
3. Đi từ **vét cạn ba vòng lặp** → **tính sẵn tính chẵn lẻ của từng số** → **gom mỗi mảng thành 2 "ngăn" (chẵn/lẻ) rồi nhân số lượng** → **gộp dần từng mảng (DP trên 2 trạng thái)** → **công thức đại số**.
4. Hiểu mấu chốt: **tính chẵn lẻ của số bit 1 trong `x ^ y ^ z` bằng XOR của ba tính chẵn lẻ riêng lẻ** ⇒ chỉ cần biết mỗi mảng có bao nhiêu số "chẵn bit" và bao nhiêu số "lẻ bit".
5. Tránh các lỗi kinh điển: dùng `set` làm mất phần tử trùng, kiểm tra "giá trị XOR chẵn" thay vì "số bit 1 chẵn", **cộng** thay vì **nhân** số lượng, và XOR các số bit 1 thay vì XOR các tính chẵn lẻ của chúng.

## 2. Đề bài & ví dụ

Cho ba mảng số nguyên `a`, `b`, `c`. Hãy trả về **số bộ ba** `(a[i], b[j], c[k])` sao cho phép **XOR bit** (bitwise XOR) của ba phần tử trong bộ ba có **số bit 1 là số chẵn**.

Lưu ý: bộ ba được tính **theo chỉ số** `(i, j, k)`. Hai bộ ba cùng giá trị nhưng khác chỉ số vẫn là **hai bộ ba khác nhau**.

**Ví dụ 1**

- Input: `a = [1]`, `b = [2]`, `c = [3]`
- Output: `1`
- Giải thích: bộ ba duy nhất là `(1, 2, 3)`. `1 XOR 2 XOR 3 = 0` (nhị phân `00`), có `0` bit 1 — là số chẵn, nên đếm `1`.

**Ví dụ 2**

- Input: `a = [1,1]`, `b = [2,3]`, `c = [1,5]`
- Output: `4`
- Giải thích: bốn bộ ba thỏa mãn là `(a[0], b[1], c[0])` và `(a[1], b[1], c[0])` có XOR `1 XOR 3 XOR 1 = 3` (nhị phân `011`, hai bit 1), cùng `(a[0], b[0], c[1])` và `(a[1], b[0], c[1])` có XOR `1 XOR 2 XOR 5 = 6` (nhị phân `110`, hai bit 1).

**Ràng buộc:** `1 <= a.length, b.length, c.length <= 100`; `0 <= a[i], b[i], c[i] <= 100`.

(Hai ví dụ trên theo hiểu biết về đề gốc; nếu có Premium nên đối chiếu lại.)

In [1]:
from typing import List
from collections import Counter
from itertools import product
import random

examples = [
    ([1], [2], [3], 1),
    ([1, 1], [2, 3], [1, 5], 4),
]
for a, b, c, expected in examples:
    print("a =", a, "b =", b, "c =", c, "-> kỳ vọng", expected)

a = [1] b = [2] c = [3] -> kỳ vọng 1
a = [1, 1] b = [2, 3] c = [1, 5] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. XOR (`^`): hai bit giống nhau cho `0`, khác nhau cho `1`

XOR là phép toán trên **từng bit** của hai số: bit kết quả là `1` nếu hai bit **khác nhau**, là `0` nếu **giống nhau**. Ví dụ `5 = 101`, `3 = 011` ⇒ `5 ^ 3 = 110 = 6`. XOR có hai tính chất rất tiện: **giao hoán** (`x ^ y == y ^ x`) và **kết hợp** (`(x ^ y) ^ z == x ^ (y ^ z)`), nên `x ^ y ^ z` không cần ngoặc và thứ tự không quan trọng.

In [2]:
x, y = 5, 3
print(bin(x), bin(y), bin(x ^ y), x ^ y)          # 0b101 0b11 0b110 6
print(1 ^ 2 ^ 3)                                  # 0  (ví dụ 1 của đề)
print((1 ^ 2) ^ 3 == 1 ^ (2 ^ 3), 1 ^ 2 == 2 ^ 1)  # True True

0b101 0b11 0b110 6
0
True True


### 3.2. Đếm bit 1 (set bits, còn gọi là popcount)

`bin(x)` đổi số sang chuỗi nhị phân dạng `'0b101'`; `.count("1")` đếm các ký tự `'1'`. Từ Python 3.10 có sẵn `x.bit_count()` làm đúng việc đó (nhanh hơn). Cả hai cho cùng một kết quả với số nguyên không âm.

In [3]:
for x in [0, 1, 2, 3, 5, 6, 100]:
    print(x, bin(x), "-> số bit 1:", bin(x).count("1"), x.bit_count())
# 0 0b0 -> 0 0
# 100 = 0b1100100 -> 3

0 0b0 -> số bit 1: 0 0
1 0b1 -> số bit 1: 1 1
2 0b10 -> số bit 1: 1 1
3 0b11 -> số bit 1: 2 2
5 0b101 -> số bit 1: 2 2
6 0b110 -> số bit 1: 2 2
100 0b1100100 -> số bit 1: 3 3


### 3.3. Chẵn / lẻ: `% 2` và `& 1`

Một số là **chẵn** nếu chia hết cho 2. Với số nguyên không âm, `x % 2` và `x & 1` đều cho `0` (chẵn) hoặc `1` (lẻ) — `x & 1` chỉ đọc **bit cuối cùng** của `x`. Trong bài này ta cần chẵn lẻ **của số bit 1**, tức `x.bit_count() & 1`. Gọi đó là **parity** (tính chẵn lẻ): parity `0` nghĩa là số bit 1 chẵn, parity `1` nghĩa là lẻ.

In [4]:
def parity(x):
    return x.bit_count() & 1

for x in [0, 1, 2, 3, 7, 100]:
    print(f"x={x:>3}  nhị phân={bin(x):>10}  số bit 1={x.bit_count()}  parity={parity(x)}")

x=  0  nhị phân=       0b0  số bit 1=0  parity=0
x=  1  nhị phân=       0b1  số bit 1=1  parity=1
x=  2  nhị phân=      0b10  số bit 1=1  parity=1
x=  3  nhị phân=      0b11  số bit 1=2  parity=0
x=  7  nhị phân=     0b111  số bit 1=3  parity=1
x=100  nhị phân= 0b1100100  số bit 1=3  parity=1


### 3.4. Bổ đề then chốt: parity của XOR = XOR của các parity

Với hai số `x`, `y`: mỗi vị trí bit mà **cả hai cùng là 1** làm XOR **mất đi hai bit 1** (hai bit `1` giống nhau cho `0`), còn vị trí chỉ một số có bit 1 thì giữ lại đúng một bit 1. Do đó

`popcount(x ^ y) = popcount(x) + popcount(y) − 2 · popcount(x & y)`.

Vì `2 · (...)` là số chẵn nên **popcount(x ^ y) và popcount(x) + popcount(y) cùng chẵn lẻ**, tức `parity(x ^ y) = parity(x) ^ parity(y)`. Áp dụng hai lần: `parity(x ^ y ^ z) = parity(x) ^ parity(y) ^ parity(z)`. Hãy kiểm chứng bằng code cho mọi cặp trong `0..127`.

In [5]:
for x in range(128):
    for y in range(128):
        assert (x ^ y).bit_count() == x.bit_count() + y.bit_count() - 2 * (x & y).bit_count()
        assert parity(x ^ y) == parity(x) ^ parity(y)
print("Công thức popcount(x^y) = popcount(x) + popcount(y) - 2*popcount(x&y) đúng với mọi cặp 0..127")
for x, y, z in [(1, 2, 3), (1, 3, 1), (1, 2, 5)]:
    print((x, y, z), "parity(XOR) =", parity(x ^ y ^ z), "| parity(x)^parity(y)^parity(z) =", parity(x) ^ parity(y) ^ parity(z))

Công thức popcount(x^y) = popcount(x) + popcount(y) - 2*popcount(x&y) đúng với mọi cặp 0..127
(1, 2, 3) parity(XOR) = 0 | parity(x)^parity(y)^parity(z) = 0
(1, 3, 1) parity(XOR) = 0 | parity(x)^parity(y)^parity(z) = 0
(1, 2, 5) parity(XOR) = 0 | parity(x)^parity(y)^parity(z) = 0


### 3.5. Đếm bằng list 2 phần tử hoặc `Counter`

Vì parity chỉ có hai giá trị `0` và `1`, ta có thể đếm bằng list `cnt = [0, 0]`: `cnt[p] += 1` nghĩa là "thêm một số có parity `p`". Cũng có thể dùng `Counter` (từ điển đếm), khi tra khóa chưa có thì trả `0` thay vì báo lỗi.

In [6]:
a = [1, 1]
cnt = [0, 0]
for x in a:
    cnt[parity(x)] += 1
print(cnt)                                         # [0, 2]: 0 số parity 0, 2 số parity 1

cnt2 = Counter(parity(x) for x in a)
print(cnt2, cnt2[0], cnt2[1])                       # Counter({1: 2}) 0 2

[0, 2]
Counter({1: 2}) 0 2


### 3.6. Quy tắc nhân và `itertools.product`

Nếu có `p` cách chọn phần tử đầu, `q` cách chọn phần tử thứ hai, `r` cách chọn phần tử thứ ba, thì có `p · q · r` bộ ba. Trong bài này: nếu mảng `a` có `2` số parity `1`, mảng `b` có `1` số parity `0`, mảng `c` có `1` số parity `1`, thì có `2 · 1 · 1 = 2` bộ ba kiểu `(1, 0, 1)`. `itertools.product(range(2), repeat=3)` liệt kê tám bộ parity `(i, j, k)` có thể có.

In [7]:
print(2 * 1 * 1)                                    # 2
for i, j, k in product(range(2), repeat=3):
    print((i, j, k), "XOR parity =", i ^ j ^ k, "-> chẵn" if i ^ j ^ k == 0 else "-> lẻ")

2
(0, 0, 0) XOR parity = 0 -> chẵn
(0, 0, 1) XOR parity = 1 -> lẻ
(0, 1, 0) XOR parity = 1 -> lẻ
(0, 1, 1) XOR parity = 0 -> chẵn
(1, 0, 0) XOR parity = 1 -> lẻ
(1, 0, 1) XOR parity = 0 -> chẵn
(1, 1, 0) XOR parity = 0 -> chẵn
(1, 1, 1) XOR parity = 1 -> lẻ


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `a = [1,1]`, `b = [2,3]`, `c = [1,5]` (ví dụ 2), liệt kê **cả 8 bộ ba** `(i, j, k)` và tính số bit 1 của XOR:

| bộ ba (giá trị) | XOR | nhị phân | số bit 1 | chẵn? |
|---|---|---|---|---|
| (1, 2, 1) | 2 | 10 | 1 | không |
| (1, 2, 5) | 6 | 110 | 2 | **có** |
| (1, 3, 1) | 3 | 11 | 2 | **có** |
| (1, 3, 5) | 7 | 111 | 3 | không |
| (1, 2, 1) (a[1]) | 2 | 10 | 1 | không |
| (1, 2, 5) (a[1]) | 6 | 110 | 2 | **có** |
| (1, 3, 1) (a[1]) | 3 | 11 | 2 | **có** |
| (1, 3, 5) (a[1]) | 7 | 111 | 3 | không |

Có `4` bộ ba "chẵn" ⇒ đáp án `4`. Nhìn kỹ: kết quả **chỉ phụ thuộc vào "chẵn/lẻ số bit 1" của từng số**, chứ không phụ thuộc giá trị cụ thể. Điều này gợi ra cách gom số theo parity.

In [8]:
a, b, c = [1, 1], [2, 3], [1, 5]
total = 0
for i, x in enumerate(a):
    for j, y in enumerate(b):
        for k, z in enumerate(c):
            v = x ^ y ^ z
            even = v.bit_count() % 2 == 0
            total += even
            print(f"a[{i}]={x} b[{j}]={y} c[{k}]={z}: XOR={v} ({bin(v)}), bit 1={v.bit_count()}, chẵn? {even}")
print("tổng =", total)

a[0]=1 b[0]=2 c[0]=1: XOR=2 (0b10), bit 1=1, chẵn? False
a[0]=1 b[0]=2 c[1]=5: XOR=6 (0b110), bit 1=2, chẵn? True
a[0]=1 b[1]=3 c[0]=1: XOR=3 (0b11), bit 1=2, chẵn? True
a[0]=1 b[1]=3 c[1]=5: XOR=7 (0b111), bit 1=3, chẵn? False
a[1]=1 b[0]=2 c[0]=1: XOR=2 (0b10), bit 1=1, chẵn? False
a[1]=1 b[0]=2 c[1]=5: XOR=6 (0b110), bit 1=2, chẵn? True
a[1]=1 b[1]=3 c[0]=1: XOR=3 (0b11), bit 1=2, chẵn? True
a[1]=1 b[1]=3 c[1]=5: XOR=7 (0b111), bit 1=3, chẵn? False
tổng = 4


## 5. Cách 1 — Brute Force / baseline: thử mọi bộ ba

**Ý tưởng.** Ba vòng lặp lồng nhau duyệt mọi `(x, y, z)`; với mỗi bộ ba tính `x ^ y ^ z`, đếm bit 1 bằng `bin(...).count("1")`, nếu chẵn thì tăng đếm. Bám sát đề, không cần suy nghĩ gì thêm.

In [9]:
def triplet_count_bruteforce(a, b, c):
    count = 0
    for x in a:
        for y in b:
            for z in c:
                if bin(x ^ y ^ z).count("1") % 2 == 0:     # số bit 1 của XOR chẵn
                    count += 1
    return count


for a, b, c, expected in examples:
    print(triplet_count_bruteforce(a, b, c), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


**Độ phức tạp (Cách 1).** Gọi `p = len(a)`, `q = len(b)`, `r = len(c)` (mỗi số `<= 100`). Có `p · q · r` bộ ba (tối đa `10^6`), mỗi bộ ba tốn một lần XOR và một lần đếm bit ⇒ **Time `O(p · q · r)`**, **Space `O(1)`**.

**Nút thắt:** với `10^6` bộ ba thì Python vẫn chạy được, nhưng ta đang **tính lại cùng một việc nhiều lần**: số bit 1 của `x ^ y ^ z` chỉ cần parity, mà parity lại tách được theo từng số. Nếu mảng dài hơn (bản II của bài) thì cách này không còn nổi.

## 6. Ý tưởng thuật toán / bước đột phá

Nhờ **bổ đề ở mục 3.4**, `x ^ y ^ z` có số bit 1 chẵn ⇔ `parity(x) ^ parity(y) ^ parity(z) == 0` ⇔ số các số có parity `1` trong ba số là **số chẵn** (`0` hoặc `2`).

Vậy mỗi số chỉ cần nhớ **một bit parity**. Hai số cùng parity là "như nhau" trong bài này. Ta không cần liệt kê từng bộ ba, mà chỉ cần biết trong mỗi mảng có **bao nhiêu số parity 0** và **bao nhiêu số parity 1**, rồi áp dụng quy tắc nhân:

`đáp án = Σ cnt_a[i] · cnt_b[j] · cnt_c[k]` với mọi `(i, j, k)` có `i ^ j ^ k == 0`.

Chỉ có **bốn** tổ hợp thỏa: `(0,0,0)`, `(0,1,1)`, `(1,0,1)`, `(1,1,0)`. Độ phức tạp tụt xuống `O(p + q + r)`.

In [10]:
a, b, c = [1, 1], [2, 3], [1, 5]
cnt_a, cnt_b, cnt_c = [0, 0], [0, 0], [0, 0]
for arr, cnt in ((a, cnt_a), (b, cnt_b), (c, cnt_c)):
    for x in arr:
        cnt[parity(x)] += 1
print("cnt_a =", cnt_a, "cnt_b =", cnt_b, "cnt_c =", cnt_c)
ans = 0
for i, j, k in product(range(2), repeat=3):
    if i ^ j ^ k == 0:
        part = cnt_a[i] * cnt_b[j] * cnt_c[k]
        print(f"(i,j,k)=({i},{j},{k}): {cnt_a[i]} * {cnt_b[j]} * {cnt_c[k]} = {part}")
        ans += part
print("đáp án =", ans)

cnt_a = [0, 2] cnt_b = [1, 1] cnt_c = [1, 1]
(i,j,k)=(0,0,0): 0 * 1 * 1 = 0
(i,j,k)=(0,1,1): 0 * 1 * 1 = 0
(i,j,k)=(1,0,1): 2 * 1 * 1 = 2
(i,j,k)=(1,1,0): 2 * 1 * 1 = 2
đáp án = 4


## 7. Các cách tiếp theo

### Cách 2 — Tính sẵn parity của từng số, rồi vẫn ba vòng lặp

**Giải thích từng đoạn.**

1. `pa = [x.bit_count() & 1 for x in a]` (tương tự `pb`, `pc`): mỗi số thành một bit `0/1`.
2. Ba vòng lặp lồng nhau trên các parity: nếu `u ^ v ^ w == 0` thì tăng đếm.

Cách này vẫn là `O(p · q · r)` nhưng mỗi bước chỉ là XOR trên số bé, không phải đếm bit — bước đệm để thấy rõ "chỉ cần parity".

In [11]:
def triplet_count_parity_lists(a, b, c):
    pa = [x.bit_count() & 1 for x in a]
    pb = [x.bit_count() & 1 for x in b]
    pc = [x.bit_count() & 1 for x in c]
    count = 0
    for u in pa:
        for v in pb:
            for w in pc:
                if u ^ v ^ w == 0:
                    count += 1
    return count


for a, b, c, expected in examples:
    print(triplet_count_parity_lists(a, b, c), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionParityLists:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        pa = [x.bit_count() & 1 for x in a]
        pb = [x.bit_count() & 1 for x in b]
        pc = [x.bit_count() & 1 for x in c]
        count = 0
        for u in pa:
            for v in pb:
                for w in pc:
                    if u ^ v ^ w == 0:
                        count += 1
        return count


print(SolutionParityLists().tripletCount([1, 1], [2, 3], [1, 5]))      # 4

4


**Độ phức tạp Cách 2:** **Time `O(p · q · r)`** (hằng số nhỏ hơn Cách 1), **Space `O(p + q + r)`** cho ba list parity.

### Cách 3 — Gom mỗi mảng thành 2 ngăn rồi nhân (cách nên nộp)

**Giải thích từng đoạn.**

1. Hàm phụ `parity_count(nums)` trả `cnt = [số_có_parity_0, số_có_parity_1]`.
2. Với ba mảng ta có `ca`, `cb`, `cc`.
3. Duyệt `i, j, k` trong `{0, 1}`; nếu `i ^ j ^ k == 0` thì cộng `ca[i] * cb[j] * cc[k]`.

Chỉ `8` phép kiểm tra cố định, cộng thêm ba lượt quét mảng.

In [13]:
def parity_count(nums):
    cnt = [0, 0]
    for x in nums:
        cnt[x.bit_count() & 1] += 1
    return cnt


def triplet_count_buckets(a, b, c):
    ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
    ans = 0
    for i in range(2):
        for j in range(2):
            for k in range(2):
                if i ^ j ^ k == 0:                    # parity XOR chẵn
                    ans += ca[i] * cb[j] * cc[k]       # quy tắc nhân
    return ans


for a, b, c, expected in examples:
    print(triplet_count_buckets(a, b, c), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionBuckets:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        def parity_count(nums: List[int]) -> List[int]:
            cnt = [0, 0]
            for x in nums:
                cnt[x.bit_count() & 1] += 1
            return cnt

        ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
        ans = 0
        for i in range(2):
            for j in range(2):
                for k in range(2):
                    if i ^ j ^ k == 0:
                        ans += ca[i] * cb[j] * cc[k]
        return ans


print(SolutionBuckets().tripletCount([1], [2], [3]))      # 1

1


**Độ phức tạp Cách 3:** **Time `O(p + q + r)`** (ba lượt quét + `8` bước cố định), **Space `O(1)`** (ba list 2 phần tử).

### Cách 4 — Gộp dần từng mảng (DP trên 2 trạng thái)

**Giải thích từng đoạn.**

1. Giữ `ways[p]` = số cách chọn các phần tử **đã xử lý** sao cho parity XOR của chúng là `p`. Lúc đầu chưa chọn gì: XOR rỗng bằng `0`, parity `0` ⇒ `ways = [1, 0]`.
2. Gộp thêm một mảng với bảng đếm `cnt`: `new[p ^ q] += ways[p] * cnt[q]`.
3. Sau khi gộp cả ba mảng, đáp án là `ways[0]`.

Ưu điểm: viết một lần dùng cho **bất kỳ số mảng nào** (không hard-code ba vòng `i, j, k`).

In [15]:
def triplet_count_fold(a, b, c):
    ways = [1, 0]                                  # chưa chọn gì: parity 0 có 1 cách
    for arr in (a, b, c):
        cnt = parity_count(arr)
        new = [0, 0]
        for p in range(2):
            for q in range(2):
                new[p ^ q] += ways[p] * cnt[q]
        ways = new
    return ways[0]


for a, b, c, expected in examples:
    print(triplet_count_fold(a, b, c), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionFold:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        ways = [1, 0]
        for arr in (a, b, c):
            cnt = [0, 0]
            for x in arr:
                cnt[x.bit_count() & 1] += 1
            new = [0, 0]
            for p in range(2):
                for q in range(2):
                    new[p ^ q] += ways[p] * cnt[q]
            ways = new
        return ways[0]


print(SolutionFold().tripletCount([1, 1], [2, 3], [1, 5]))      # 4

4


**Độ phức tạp Cách 4:** **Time `O(p + q + r)`**, **Space `O(1)`**.

### Cách 5 — Công thức đại số: `(tổng + tích các hiệu) / 2`

**Giải thích từng đoạn.**

1. Gán mỗi số dấu `+1` nếu parity `0` (chẵn) và `−1` nếu parity `1` (lẻ). Với một bộ ba, **tích ba dấu** là `+1` khi số số lẻ là chẵn (parity XOR `0`) và `−1` khi lẻ.
2. Cộng tích đó trên **mọi** bộ ba. Nhờ phép phân phối, `Σ (sa · sb · sc) = (Σ sa) · (Σ sb) · (Σ sc)`. Đặt `Da = cnt_a[0] − cnt_a[1]` (và `Db`, `Dc`). Vậy `chẵn − lẻ = Da · Db · Dc`.
3. Mà `chẵn + lẻ = p · q · r`. Cộng hai đẳng thức: `chẵn = (p·q·r + Da·Db·Dc) / 2`.

Công thức đẹp nhưng khó nhìn bằng mắt hơn Cách 3 — dùng để kiểm chứng chéo.

In [17]:
def triplet_count_formula(a, b, c):
    ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
    total = len(a) * len(b) * len(c)
    diff = (ca[0] - ca[1]) * (cb[0] - cb[1]) * (cc[0] - cc[1])      # chẵn - lẻ
    return (total + diff) // 2


for a, b, c, expected in examples:
    print(triplet_count_formula(a, b, c), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionFormula:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        def diff(nums: List[int]) -> int:
            return sum(1 - 2 * (x.bit_count() & 1) for x in nums)   # +1 nếu parity 0, -1 nếu parity 1

        return (len(a) * len(b) * len(c) + diff(a) * diff(b) * diff(c)) // 2


print(SolutionFormula().tripletCount([1, 1], [2, 3], [1, 5]))      # 4

4


**Độ phức tạp Cách 5:** **Time `O(p + q + r)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace **Cách 3** (gom 2 ngăn) trên ví dụ 2: `a = [1,1]`, `b = [2,3]`, `c = [1,5]`. Đầu tiên quét từng mảng để lập `cnt`, sau đó duyệt tám tổ hợp `(i, j, k)`.

| mảng | số | bit 1 | parity | `cnt` sau khi xử lý |
|---|---|---|---|---|
| a | 1 | 1 | 1 | `[0, 1]` |
| a | 1 | 1 | 1 | `[0, 2]` |
| b | 2 | 1 | 1 | `[0, 1]` |
| b | 3 | 2 | 0 | `[1, 1]` |
| c | 1 | 1 | 1 | `[0, 1]` |
| c | 5 | 2 | 0 | `[1, 1]` |

| (i,j,k) | `i^j^k` | tích `ca[i]·cb[j]·cc[k]` | đếm? |
|---|---|---|---|
| (0,0,0) | 0 | 0·1·1 = 0 | có, cộng 0 |
| (0,0,1) | 1 | — | không |
| (0,1,0) | 1 | — | không |
| (0,1,1) | 0 | 0·1·1 = 0 | có, cộng 0 |
| (1,0,0) | 1 | — | không |
| (1,0,1) | 0 | 2·1·1 = 2 | có, cộng 2 |
| (1,1,0) | 0 | 2·1·1 = 2 | có, cộng 2 |
| (1,1,1) | 1 | — | không |

Tổng `0 + 0 + 2 + 2 = 4`.

In [19]:
def trace_buckets(a, b, c):
    print(f"a={a} b={b} c={c}")
    counts = []
    for name, arr in (("a", a), ("b", b), ("c", c)):
        cnt = [0, 0]
        for x in arr:
            p = x.bit_count() & 1
            cnt[p] += 1
            print(f"  mảng {name}: x={x} ({bin(x)}), bit 1={x.bit_count()}, parity={p}, cnt={cnt}")
        counts.append(cnt)
    ca, cb, cc = counts
    ans = 0
    for i in range(2):
        for j in range(2):
            for k in range(2):
                if i ^ j ^ k == 0:
                    part = ca[i] * cb[j] * cc[k]
                    ans += part
                    print(f"  (i,j,k)=({i},{j},{k}) XOR parity 0: {ca[i]}*{cb[j]}*{cc[k]} = {part}, tổng = {ans}")
                else:
                    print(f"  (i,j,k)=({i},{j},{k}) XOR parity 1: bỏ qua")
    return ans


assert trace_buckets([1, 1], [2, 3], [1, 5]) == 4

a=[1, 1] b=[2, 3] c=[1, 5]
  mảng a: x=1 (0b1), bit 1=1, parity=1, cnt=[0, 1]
  mảng a: x=1 (0b1), bit 1=1, parity=1, cnt=[0, 2]
  mảng b: x=2 (0b10), bit 1=1, parity=1, cnt=[0, 1]
  mảng b: x=3 (0b11), bit 1=2, parity=0, cnt=[1, 1]
  mảng c: x=1 (0b1), bit 1=1, parity=1, cnt=[0, 1]
  mảng c: x=5 (0b101), bit 1=2, parity=0, cnt=[1, 1]
  (i,j,k)=(0,0,0) XOR parity 0: 0*1*1 = 0, tổng = 0
  (i,j,k)=(0,0,1) XOR parity 1: bỏ qua
  (i,j,k)=(0,1,0) XOR parity 1: bỏ qua
  (i,j,k)=(0,1,1) XOR parity 0: 0*1*1 = 0, tổng = 0
  (i,j,k)=(1,0,0) XOR parity 1: bỏ qua
  (i,j,k)=(1,0,1) XOR parity 0: 2*1*1 = 2, tổng = 2
  (i,j,k)=(1,1,0) XOR parity 0: 2*1*1 = 2, tổng = 4
  (i,j,k)=(1,1,1) XOR parity 1: bỏ qua


In [20]:
def trace_fold(a, b, c):
    ways = [1, 0]
    print("bắt đầu ways =", ways)
    for name, arr in (("a", a), ("b", b), ("c", c)):
        cnt = parity_count(arr)
        new = [0, 0]
        for p in range(2):
            for q in range(2):
                new[p ^ q] += ways[p] * cnt[q]
        print(f"gộp {name}: cnt={cnt}, ways {ways} -> {new}")
        ways = new
    print("đáp án = ways[0] =", ways[0])
    return ways[0]


assert trace_fold([1, 1], [2, 3], [1, 5]) == 4

bắt đầu ways = [1, 0]
gộp a: cnt=[0, 2], ways [1, 0] -> [0, 2]
gộp b: cnt=[1, 1], ways [0, 2] -> [2, 2]
gộp c: cnt=[1, 1], ways [2, 2] -> [4, 4]
đáp án = ways[0] = 4


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (gom 2 ngăn rồi nhân): dễ đọc, `O(p + q + r)`, không lệ thuộc công thức.

In [21]:
class Solution:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        def parity_count(nums: List[int]) -> List[int]:
            cnt = [0, 0]
            for x in nums:
                cnt[x.bit_count() & 1] += 1       # parity của số bit 1
            return cnt

        ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
        ans = 0
        for i in range(2):
            for j in range(2):
                for k in range(2):
                    if i ^ j ^ k == 0:           # số bit 1 của XOR là chẵn
                        ans += ca[i] * cb[j] * cc[k]
        return ans


sol = Solution()
for a, b, c, expected in examples:
    print(a, b, c, "->", sol.tripletCount(a, b, c), "(kỳ vọng", expected, ")")

[1] [2] [3] -> 1 (kỳ vọng 1 )
[1, 1] [2, 3] [1, 5] -> 4 (kỳ vọng 4 )


## 10. Vì sao thuật toán đúng

- **Bổ đề parity.** `popcount(x ^ y) = popcount(x) + popcount(y) − 2·popcount(x & y)`, nên `popcount(x ^ y)` và `popcount(x) + popcount(y)` cùng chẵn lẻ. Lặp lại cho số thứ ba: số bit 1 của `x ^ y ^ z` chẵn ⇔ `parity(x) ^ parity(y) ^ parity(z) == 0`.
- **Mỗi bộ ba thuộc đúng một ngăn.** Bộ ba `(a[i], b[j], c[k])` có đúng một bộ parity `(p, q, r)` ∈ `{0,1}^3`. Các ngăn không giao nhau và phủ hết mọi bộ ba, nên cộng theo ngăn không đếm trùng hay sót.
- **Quy tắc nhân.** Số bộ ba thuộc ngăn `(i, j, k)` là `ca[i] · cb[j] · cc[k]`: chọn tự do một số trong `ca[i]` số của `a`, một số trong `cb[j]` số của `b`, một số trong `cc[k]` số của `c`. Các phần tử **trùng giá trị** vẫn được đếm riêng vì đề đếm theo chỉ số.
- **Chỉ lấy ngăn chẵn.** Ta cộng đúng những ngăn có `i ^ j ^ k == 0`, tức parity XOR bằng `0` — đúng điều kiện "số bit 1 chẵn" của đề.

## 11. Phân tích Time / Space Complexity

Gọi `p = len(a)`, `q = len(b)`, `r = len(c)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force ba vòng | `O(p·q·r)` | `O(1)` |
| 2. Parity sẵn + ba vòng | `O(p·q·r)` | `O(p+q+r)` |
| 3. Gom 2 ngăn rồi nhân | `O(p+q+r)` | `O(1)` |
| 4. Gộp dần (DP 2 trạng thái) | `O(p+q+r)` | `O(1)` |
| 5. Công thức đại số | `O(p+q+r)` | `O(1)` |

Với `p, q, r <= 100`, cách 1 tối đa `10^6` bước (vẫn qua), cách 3 chỉ cỡ `300 + 8` bước. Đáp án lớn nhất là `10^6`, vừa trong `int`.

## 12. Bộ test

Ta gom mọi cách vào `impls` rồi so với kỳ vọng.

In [22]:
impls = {
    "bruteforce": triplet_count_bruteforce,
    "parity_lists": triplet_count_parity_lists,
    "buckets": triplet_count_buckets,
    "fold": triplet_count_fold,
    "formula": triplet_count_formula,
    "leetcode": Solution().tripletCount,
    "leetcode_c2": SolutionParityLists().tripletCount,
    "leetcode_c3": SolutionBuckets().tripletCount,
    "leetcode_c4": SolutionFold().tripletCount,
    "leetcode_c5": SolutionFormula().tripletCount,
}


def check_all(a, b, c, expected, label):
    for key, fn in impls.items():
        actual = fn(a, b, c)
        assert actual == expected, (label, key, a, b, c, actual, expected)
    print("PASS", label)


for idx, (a, b, c, expected) in enumerate(examples, 1):
    check_all(a, b, c, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all([0], [0], [0], 1, "toàn số 0: XOR = 0, 0 bit 1 (chẵn)")
check_all([1], [0], [0], 0, "XOR = 1, một bit 1 (lẻ)")
check_all([1], [1], [0], 1, "1 ^ 1 ^ 0 = 0, chẵn")
check_all([3], [1], [1], 1, "3 ^ 1 ^ 1 = 3 (hai bit 1), chẵn")
check_all([100], [100], [100], 0, "100 ^ 100 ^ 100 = 100 = 1100100 (ba bit 1), lẻ")
check_all([3] * 100, [3] * 100, [3] * 100, 10 ** 6, "100^3 bộ ba cùng chẵn")
check_all([0] * 100, [0] * 100, [0] * 100, 10 ** 6, "toàn số 0, kích thước tối đa")
check_all([1] * 100, [1] * 100, [1] * 100, 0, "toàn số 1: XOR luôn bằng 1 (lẻ)")
check_all([1, 2, 3], [4], [5, 6], 4, "bài tính tay trong mục 17")
check_all(list(range(0, 101)), [7], [9], sum(1 for x in range(101) if (x ^ 7 ^ 9).bit_count() % 2 == 0), "a = 0..100 (101 số, chỉ để kiểm tra tay - ngoài ràng buộc độ dài)")

PASS toàn số 0: XOR = 0, 0 bit 1 (chẵn)
PASS XOR = 1, một bit 1 (lẻ)
PASS 1 ^ 1 ^ 0 = 0, chẵn
PASS 3 ^ 1 ^ 1 = 3 (hai bit 1), chẵn
PASS 100 ^ 100 ^ 100 = 100 = 1100100 (ba bit 1), lẻ


PASS 100^3 bộ ba cùng chẵn


PASS toàn số 0, kích thước tối đa


PASS toàn số 1: XOR luôn bằng 1 (lẻ)
PASS bài tính tay trong mục 17
PASS a = 0..100 (101 số, chỉ để kiểm tra tay - ngoài ràng buộc độ dài)


### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi phổ biến (chi tiết ở mục 13): (a) dùng `set` làm mất phần tử trùng; (b) kiểm tra **giá trị** XOR chẵn thay vì số bit 1 chẵn; (c) **cộng** thay vì **nhân** số lượng; (d) XOR các **số bit 1** thay vì XOR các **parity**.

In [24]:
def wrong_set(a, b, c):
    return triplet_count_buckets(list(set(a)), list(set(b)), list(set(c)))            # SAI: mất bản sao


def wrong_value_even(a, b, c):
    return sum((x ^ y ^ z) % 2 == 0 for x in a for y in b for z in c)                # SAI: giá trị chẵn, không phải số bit 1 chẵn


def wrong_add(a, b, c):
    ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
    return sum(ca[i] + cb[j] + cc[k] for i in range(2) for j in range(2) for k in range(2) if i ^ j ^ k == 0)   # SAI: cộng


def wrong_xor_of_counts(a, b, c):
    return sum((x.bit_count() ^ y.bit_count() ^ z.bit_count()) == 0 for x in a for y in b for z in c)             # SAI


print(wrong_set([1, 1], [2, 3], [1, 5]), "!=", triplet_count_buckets([1, 1], [2, 3], [1, 5]))
print(wrong_value_even([1], [2], [0]), "!=", triplet_count_buckets([1], [2], [0]))
print(wrong_add([1, 1], [2, 3], [1, 5]), "!=", triplet_count_buckets([1, 1], [2, 3], [1, 5]))
print(wrong_xor_of_counts([3], [1], [1]), "!=", triplet_count_buckets([3], [1], [1]))
assert wrong_set([1, 1], [2, 3], [1, 5]) != 4
assert wrong_value_even([1], [2], [0]) != triplet_count_buckets([1], [2], [0])
assert wrong_add([1, 1], [2, 3], [1, 5]) != 4
assert wrong_xor_of_counts([3], [1], [1]) != 1

2 != 4
0 != 1
12 != 4
0 != 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: độ dài mỗi mảng `1..100`, giá trị `0..100`. Ta kiểm tra: (1) **ngẫu nhiên** nhiều bộ ba mảng nhỏ so cả `10` cài đặt; (2) vài bộ **kích thước tối đa**; (3) **vét cạn** mọi mảng một phần tử với giá trị `0..20`.

In [25]:
rng = random.Random(3199)
for trial in range(400):
    p, q, r = rng.randint(1, 15), rng.randint(1, 15), rng.randint(1, 15)
    a = [rng.randint(0, 100) for _ in range(p)]
    b = [rng.randint(0, 100) for _ in range(q)]
    c = [rng.randint(0, 100) for _ in range(r)]
    ref = triplet_count_bruteforce(a, b, c)
    for key, fn in impls.items():
        assert fn(a, b, c) == ref, (key, a, b, c)
print("PASS 400 bộ ba mảng ngẫu nhiên nhỏ")

for trial in range(3):
    a = [rng.randint(0, 100) for _ in range(100)]
    b = [rng.randint(0, 100) for _ in range(100)]
    c = [rng.randint(0, 100) for _ in range(100)]
    ref = triplet_count_bruteforce(a, b, c)
    for key, fn in impls.items():
        assert fn(a, b, c) == ref, (key, "kích thước 100")
print("PASS 3 bộ ba mảng kích thước tối đa 100")

count = 0
for x, y, z in product(range(21), repeat=3):
    assert triplet_count_buckets([x], [y], [z]) == int((x ^ y ^ z).bit_count() % 2 == 0)
    count += 1
print("PASS vét cạn", count, "bộ (x, y, z) mỗi mảng một phần tử")

PASS 400 bộ ba mảng ngẫu nhiên nhỏ


PASS 3 bộ ba mảng kích thước tối đa 100
PASS vét cạn 9261 bộ (x, y, z) mỗi mảng một phần tử


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `set` để "bỏ trùng" mảng

In [26]:
def sai_dung_set(a, b, c):
    a, b, c = set(a), set(b), set(c)               # SAI: đề đếm theo chỉ số, bản sao vẫn tính
    return sum(bin(x ^ y ^ z).count("1") % 2 == 0 for x in a for y in b for z in c)


print("sai:", sai_dung_set([1, 1], [2, 3], [1, 5]), "| đúng:", Solution().tripletCount([1, 1], [2, 3], [1, 5]))

sai: 2 | đúng: 4


**Vì sao sai:** `a = [1, 1]` có hai chỉ số khác nhau, nên `(a[0], ...)` và `(a[1], ...)` là **hai** bộ ba riêng. `set` gộp hai số `1` thành một và làm kết quả bị chia đôi (`2` thay vì `4`). **Cách sửa:** giữ nguyên danh sách, đếm theo `cnt[p] += 1` cho **từng** phần tử.

### Lỗi 2: Kiểm tra "giá trị XOR chẵn" thay vì "số bit 1 chẵn"

In [27]:
def sai_gia_tri_chan(a, b, c):
    return sum((x ^ y ^ z) % 2 == 0 for x in a for y in b for z in c)    # SAI: đang kiểm tra bit cuối


# 1 ^ 2 ^ 0 = 3 = 11: giá trị lẻ nhưng có hai bit 1 (chẵn)
print("sai:", sai_gia_tri_chan([1], [2], [0]), "| đúng:", Solution().tripletCount([1], [2], [0]))
print("sai:", sai_gia_tri_chan([3], [1], [1]), "| đúng:", Solution().tripletCount([3], [1], [1]))

sai: 0 | đúng: 1
sai: 0 | đúng: 1


**Vì sao sai:** "số chẵn" (`v % 2 == 0`) chỉ nhìn **bit cuối**. Đề hỏi **số lượng bit 1** của `v` có chẵn hay không (`v.bit_count() % 2 == 0`). Ví dụ `v = 3 = 11` có hai bit 1 (chẵn) nhưng `3` là số lẻ. **Cách sửa:** luôn đếm bit 1 trước rồi mới lấy chẵn lẻ.

### Lỗi 3: Cộng thay vì nhân số lượng ở các ngăn

In [28]:
def sai_cong_thay_nhan(a, b, c):
    ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
    ans = 0
    for i in range(2):
        for j in range(2):
            for k in range(2):
                if i ^ j ^ k == 0:
                    ans += ca[i] + cb[j] + cc[k]         # SAI: phải nhân
    return ans


print("sai:", sai_cong_thay_nhan([1, 1], [2, 3], [1, 5]), "| đúng:", Solution().tripletCount([1, 1], [2, 3], [1, 5]))

sai: 12 | đúng: 4


**Vì sao sai:** mỗi bộ ba là một cách **chọn đồng thời** một số từ mỗi mảng ⇒ theo quy tắc nhân, số cách là `ca[i] · cb[j] · cc[k]`. Phép cộng đếm sai và tăng sai theo cấp số cộng. **Cách sửa:** dùng `*`.

### Lỗi 4: XOR các số bit 1 thay vì XOR các parity

In [29]:
def sai_xor_so_bit(a, b, c):
    return sum((x.bit_count() ^ y.bit_count() ^ z.bit_count()) == 0 for x in a for y in b for z in c)   # SAI


# 3 = 11 (2 bit), 1 = 1 (1 bit), 1 = 1 (1 bit): 3 ^ 1 ^ 1 = 3 có 2 bit 1 (chẵn)
print("sai:", sai_xor_so_bit([3], [1], [1]), "| đúng:", Solution().tripletCount([3], [1], [1]))

sai: 0 | đúng: 1


**Vì sao sai:** bổ đề chỉ đúng với **parity** (`& 1`), không đúng với **chính số bit 1**. Ở ví dụ trên: `2 ^ 1 ^ 1 = 2 ≠ 0` nên code sai coi là lẻ, nhưng tổng `2 + 1 + 1 = 4` chẵn. **Cách sửa:** rút gọn mỗi số bit 1 về `& 1` **trước** khi XOR, hoặc dùng tổng `% 2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số bộ/cặp thỏa điều kiện về **chẵn lẻ**" ⇒ đừng liệt kê, hãy **gom theo chẵn/lẻ** rồi dùng quy tắc nhân.
- Điều kiện về **số bit 1 của XOR** ⇒ parity của XOR = XOR của các parity (dùng `& 1`). Mẹo nhớ: `x ^ y` chỉ **xóa các cặp bit 1 trùng nhau**, mỗi lần xóa đi `2` bit 1, không đổi chẵn lẻ.
- Nhìn thấy **ba mảng độc lập** + điều kiện chỉ phụ thuộc "loại" của từng phần tử ⇒ nghĩ đến `cnt_a`, `cnt_b`, `cnt_c`, không cần ba vòng lặp.
- Muốn mở rộng cho `k` mảng: dùng **gộp dần** `new[p ^ q] += ways[p] * cnt[q]` (Cách 4).
- Luôn nhớ đề đếm theo **chỉ số**: các phần tử trùng giá trị vẫn được tính riêng.

## 15. Biến thể của bài

1. **Cặp thay vì bộ ba:** đếm số cặp `(a[i], b[j])` mà XOR có số bit 1 chẵn (`ea·eb + oa·ob`).
2. **Số bit 1 lẻ:** đếm số bộ ba có parity XOR `1` (`total − even`).
3. **`k` mảng tổng quát:** gộp dần hoặc công thức `(total + ΠD)/2`.
4. **Bản II (Count Triplets with Even XOR Set Bits II, 3215):** cùng ý tưởng parity nhưng ràng buộc lớn hơn nên bắt buộc cách `O(n)`.
5. Các bài liên quan: **Number of 1 Bits (191)**, **Counting Bits (338)**, **Hamming Distance (461)**, **Sort Integers by The Number of 1 Bits (1356)**, **Single Number (136)**.

In [30]:
def count_pairs_even(a, b):
    ca, cb = parity_count(a), parity_count(b)
    return ca[0] * cb[0] + ca[1] * cb[1]


def count_triplets_odd(a, b, c):
    return len(a) * len(b) * len(c) - Solution().tripletCount(a, b, c)


def count_tuples_even(arrays):
    ways = [1, 0]
    for arr in arrays:
        cnt = parity_count(arr)
        new = [0, 0]
        for p in range(2):
            for q in range(2):
                new[p ^ q] += ways[p] * cnt[q]
        ways = new
    return ways[0]


print(count_pairs_even([1, 2], [3, 4]))                       # 2: (1,4) và (2,4) có XOR 5 và 6, mỗi số hai bit 1
print(count_triplets_odd([1, 1], [2, 3], [1, 5]))             # 8 - 4 = 4
print(count_tuples_even([[1, 1], [2, 3], [1, 5]]))            # 4 (giống bài gốc)
print(count_tuples_even([[1, 2, 3], [4, 5]]))                 # 3: ba cặp (1,4), (2,4), (3,5)

2
4
4
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi bộ ba, đếm bit XOR | `O(p·q·r)` | `O(1)` | hiểu đề, kiểm chứng |
| 2. Parity sẵn | đổi mỗi số thành 1 bit rồi vẫn ba vòng | `O(p·q·r)` | `O(p+q+r)` | cầu nối để thấy bổ đề |
| 3. Hai ngăn + nhân | `cnt[0]`, `cnt[1]` mỗi mảng | `O(p+q+r)` | `O(1)` | **nên nộp** |
| 4. Gộp dần | DP trên parity tích lũy | `O(p+q+r)` | `O(1)` | số mảng bất kỳ |
| 5. Công thức | `(tổng + ΠD)/2` | `O(p+q+r)` | `O(1)` | ngắn gọn, kiểm chứng chéo |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 ^ 3`, `bin(10)`, `(7).bit_count()`, `((6 ^ 5).bit_count()) % 2`, `Solution().tripletCount([1], [2], [3])`.

**Bài 2 (tính tay).** Với `a = [1, 2, 3]`, `b = [4]`, `c = [5, 6]`: tính parity của từng số, lập `cnt_a`, `cnt_b`, `cnt_c` rồi suy ra số bộ ba thỏa mãn. Sau đó liệt kê tay `6` bộ ba để đối chiếu.

**Bài 3 (điền TODO).** Hoàn thành `triplet_count_todo(a, b, c)` theo Cách 3.

In [31]:
def triplet_count_todo(a, b, c):
    # TODO 1: viết (hoặc dùng) hàm đếm parity: cnt[p] = số phần tử có số bit 1 chẵn/lẻ tương ứng p
    # TODO 2: lập ca, cb, cc
    # TODO 3: duyệt i, j, k trong {0, 1}; nếu i ^ j ^ k == 0 thì cộng ca[i] * cb[j] * cc[k]
    # TODO 4: trả về tổng
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_pairs_even_xor(a, b)`: đếm số cặp `(a[i], b[j])` mà `a[i] ^ b[j]` có số bit 1 chẵn, **không** dùng hai vòng lặp lồng nhau (dùng parity).

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_tuples_parity(arrays, want)` với `arrays` là danh sách `k` mảng, `want` là `0` (đếm bộ có số bit 1 chẵn) hoặc `1` (lẻ). (b) Chứng minh bằng code rằng với mọi đầu vào, `chẵn + lẻ = tích độ dài các mảng`.

In [32]:
def count_pairs_even_xor(a, b):
    # TODO: dùng parity_count rồi cộng ca[0]*cb[0] + ca[1]*cb[1]
    raise NotImplementedError


def count_tuples_parity(arrays, want):
    # TODO: gộp dần ways như Cách 4, rồi trả ways[want]
    raise NotImplementedError

# KHÔNG gọi hai hàm TODO này khi Run All.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6`, `'0b1010'`, `3`, `0`, `1`. Bài 3: đếm parity bằng `cnt[x.bit_count() & 1] += 1`.
- **Mức 2 (vừa):** Bài 2: parity `1→1`, `2→1`, `3→0`, `4→1`, `5→0`, `6→0` ⇒ `cnt_a = [1, 2]`, `cnt_b = [0, 1]`, `cnt_c = [2, 0]`. Chỉ ngăn `(1, 1, 0)` là parity XOR `0` ⇒ `2 · 1 · 2 = 4`. Bài 4: hai số cùng parity ⇒ XOR có parity `0`.
- **Mức 3 (sâu):** Bài 5(a): `ways = [1, 0]`; với mỗi mảng cập nhật `new[p ^ q] += ways[p] * cnt[q]`; trả `ways[want]`. Bài 5(b): `ways[0] + ways[1]` luôn bằng tổng độ dài nhân nhau vì mỗi bộ thuộc đúng một ngăn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert 5 ^ 3 == 6
assert bin(10) == "0b1010"
assert (7).bit_count() == 3
assert (6 ^ 5).bit_count() % 2 == 0          # 6 ^ 5 = 3 = 11 -> 2 bit -> chẵn
assert Solution().tripletCount([1], [2], [3]) == 1

# Bài 2: cnt_a = [1, 2], cnt_b = [0, 1], cnt_c = [2, 0] -> chỉ (1,1,0): 2*1*2 = 4
assert parity_count([1, 2, 3]) == [1, 2]
assert parity_count([4]) == [0, 1]
assert parity_count([5, 6]) == [2, 0]
assert Solution().tripletCount([1, 2, 3], [4], [5, 6]) == 2 * 1 * 2 == 4


# Bài 3
def triplet_count_todo_ans(a, b, c):
    ca, cb, cc = parity_count(a), parity_count(b), parity_count(c)
    ans = 0
    for i in range(2):
        for j in range(2):
            for k in range(2):
                if i ^ j ^ k == 0:
                    ans += ca[i] * cb[j] * cc[k]
    return ans


for a, b, c, expected in examples:
    assert triplet_count_todo_ans(a, b, c) == expected


# Bài 4
def count_pairs_even_xor_ans(a, b):
    ca, cb = parity_count(a), parity_count(b)
    return ca[0] * cb[0] + ca[1] * cb[1]


for a, b in [([1, 2, 3], [4, 5]), ([0], [0]), ([1], [2]), ([7, 7, 7], [1, 2, 3])]:
    brute = sum((x ^ y).bit_count() % 2 == 0 for x in a for y in b)
    assert count_pairs_even_xor_ans(a, b) == brute, (a, b)


# Bài 5
def count_tuples_parity_ans(arrays, want):
    ways = [1, 0]
    for arr in arrays:
        cnt = parity_count(arr)
        new = [0, 0]
        for p in range(2):
            for q in range(2):
                new[p ^ q] += ways[p] * cnt[q]
        ways = new
    return ways[want]


rng = random.Random(1)
for _ in range(200):
    k = rng.randint(1, 4)
    arrays = [[rng.randint(0, 100) for _ in range(rng.randint(1, 5))] for _ in range(k)]
    total = 1
    for arr in arrays:
        total *= len(arr)
    even = count_tuples_parity_ans(arrays, 0)
    odd = count_tuples_parity_ans(arrays, 1)
    assert even + odd == total
    brute_even = sum(sum(v.bit_count() for v in t) % 2 == 0 for t in product(*arrays))
    assert even == brute_even
assert count_tuples_parity_ans([[1, 1], [2, 3], [1, 5]], 0) == 4
print("Tất cả đáp án bài tập đều PASS")

Tất cả đáp án bài tập đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (gom theo parity số bit 1, nhân số lượng)

```python
class Solution:
    def tripletCount(self, a: List[int], b: List[int], c: List[int]) -> int:
        def cnt(nums):
            r = [0, 0]
            for x in nums:
                r[x.bit_count() & 1] += 1
            return r

        ca, cb, cc = cnt(a), cnt(b), cnt(c)
        return sum(ca[i] * cb[j] * cc[k]
                   for i in range(2) for j in range(2) for k in range(2)
                   if i ^ j ^ k == 0)
```

**Checklist**

- [ ] Đã kiểm tra **số bit 1** của XOR chẵn (không phải giá trị XOR chẵn)?
- [ ] Đã rút gọn mỗi số về **parity** (`& 1`) trước khi XOR/cộng?
- [ ] Đã **nhân** `ca[i] * cb[j] * cc[k]` (không cộng)?
- [ ] Giữ **phần tử trùng** (không dùng `set`), vì đề đếm theo chỉ số?
- [ ] Chỉ lấy các ngăn có `i ^ j ^ k == 0` (bốn ngăn)?
- [ ] Đã thử `[1],[2],[3]` (`1`), `[1,1],[2,3],[1,5]` (`4`), toàn số `0`, toàn số `1`?